# Histórica: combinación de trayectorias frente a una sola trayectoria

Ensayo sin recalibración, química theta_nv2 y coeficientes históricos de `configs/models/historical/gas_fit.json`, partición `train_019_021`. LAB019–021 fueron calibración previa de esos coeficientes; LAB016–018 son holdout sin reajuste. Ningún dato participa en un ajuste nuevo.

Se calculan las mismas dos trayectorias químicas que usa la histórica y se cambia únicamente la expresión que combina sus producciones:
\[
q_N=0.9552854352(\beta_G+\beta_F)X\quad\text{con nutrición},\qquad q_0=0.9552854352(\beta_G+\beta_F)X\quad\text{sin nutrición}.
\]
Para los tres casos la fuente es $q_{prod}=\mathrm{gate}\;f_{O2}\;M\;B$, con los mismos gate, O₂ y multiplicador transitorio M de la referencia:

| Caso | Expresión B |
|---|---|
| Histórica actual | $\max\{[1+(g_N-1)r]q_0+r(q_N-q_0),0\}$ |
| Sólo trayectoria con nutrición | $q_N$ |
| Sólo trayectoria sin nutrición | $q_0$ |

**Definición del cambio:** se sustituye el corchete completo de combinación. No se añade `[1+(g_N-1)r]` a las trayectorias únicas: ese factor es parte de la combinación retirada. Los valores guardados no se modifican; gN y r dejan de intervenir en esa expresión en los casos únicos. M sigue incluyendo el transitorio A=0.3, tau_r=4 h y tau_d=12 h de la referencia.

Por tanto, «sólo trayectoria sin nutrición» describe el origen de q₀; **no elimina todos los efectos nutricionales externos**. O₂, Csat(T,E,G+F) y activación se mantienen exactamente como en la histórica nutrida, sin recalcularlos a partir de otra trayectoria. Esto aísla el papel de la combinación, no compara tres formulaciones químicas completas.

## Transferencia y controles

$\dot C_d=q_{prod}-q_{gas}$, $q_{gas}=k C_d[0.05+0.95C_d/(C_d+C_{sat})]$, salida al sensor $g\,q_{gas}$. Se reutiliza el integrador implícito histórico. Cd(0)=0 en todos los casos. La activación conservada empieza a 2 g/L predichos consumidos y termina a 10; no es onset manual ni el cambio a 5 g/L que se había propuesto. Se conserva la modulación histórica de O₂.

La señal observada son medianas horarias RAW de `co2_mass_balance_lab016_021_g_l_h.csv`, ya convertida por volumen remanente. No se modifica esa conversión ni se añade filtrado a las predicciones. La referencia física y sus coeficientes corresponden a la comparación actual; no se sustituyen por los coeficientes compartidos del notebook 02 antiguo.

Las figuras de emisión separan cada alternativa frente a la histórica para evitar superponerlas todas. También se muestran fuente, Cd y métricas por laboratorio. No se modifican módulos, configuraciones, datos ni notebooks existentes. Sin exportación CSV/PNG.


In [ ]:
from pathlib import Path
import sys,json,hashlib
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/wine_fermentation').is_dir())
sys.path.insert(0,str(ROOT/'src'))
from wine_fermentation.models.historical.calibration import load,release_forward,GAS_NAMES
from wine_fermentation.comparison import observations
protected=[p for folder in ('src','configs','notebooks','results') for p in (ROOT/folder).rglob('*') if p.is_file() and '__pycache__' not in str(p) and '05_historical_single_trajectory' not in p.name]
def hashes():return {str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
before=hashes()
v=load(ROOT)
v.prepare('theta_nv2',v.thetas['theta_nv2'],tuple(f'LAB{i:03}' for i in range(16,22)))
stored=json.loads((ROOT/'configs/models/historical/gas_fit.json').read_text())['experiments']['train_019_021']
p={k:stored[k] for k in GAS_NAMES}
LABS=['LAB019','LAB020','LAB021','LAB016','LAB017','LAB018']
raw=pd.read_csv(ROOT/'data/processed/co2_mass_balance_lab016_021_g_l_h.csv',usecols=['lab','time_h','record_kind','co2_raw_g_l_h'])
observed=observations(raw)
display(pd.Series(p,name='Coeficientes fijos'))
display(pd.Series({k:v.old[k] for k in ['pulse_activity_gain','pulse_t_rise_h','A','tau_rise_h','tau_decay_h','N_ref_g_l','O2_initial_scale','O2_qmax_mg_gdw_h']}))


In [ ]:
curves={};rows=[];controls=[]
for lab in LABS:
 c=v.cache['theta_nv2',lab];t=c['time'];r=c['ramp'];activity=1+(v.old['pulse_activity_gain']-1)*r
 brackets={'Histórica':np.maximum(activity*c['q0']+r*(c['qn']-c['q0']),0),
           'Sólo con nutrición':c['qn'],'Sólo sin nutrición':c['q0']}
 original=release_forward(c,p)
 first_pulse=c['events'][0][0]
 for name,B in brackets.items():
  source=c['gate']*c['fferm']*c['multiplier']*B
  if name=='Histórica':
   assert np.allclose(source,c['source'],rtol=1e-12,atol=1e-12)
   source=c['source'].copy()  # Retain exact historical floating-point ordering.
  cache={**c,'source':source}
  pred,cd,qgas=release_forward(cache,p)
  f=pd.DataFrame(dict(qprod=source,qpred=pred,Cd=cd,qgas=qgas,qN=c['qn'],q0=c['q0']),index=t)
  curves[name,lab]=f
  balance=np.diff(cd)-np.diff(t)*(source[:-1]-qgas[1:])
  assert abs(balance).max()<1e-9 and cd.min()>=0 and qgas.min()>=-1e-10
  if name=='Histórica':
   assert np.allclose(source,c['source'],rtol=1e-12,atol=1e-12)
   assert np.array_equal(pred,original[0])
  prior=t<first_pulse
  # Prior to nutrition the two trajectories coincide up to solver roundoff.
  prior_error=np.max(abs(pred[prior]-original[0][prior]))
  assert prior_error<1e-5
  o=observed[lab];o=o[o.time_h.between(t[0],t[-1])]
  prediction=np.interp(o.time_h,t,pred);error=prediction-o.co2_raw_g_l_h.to_numpy()
  censored=np.where(o.co2_raw_g_l_h.to_numpy()<.05,np.maximum(prediction-.05,0),error)
  rows.append(dict(lab=lab,case=name,role='Calibración previa' if lab in LABS[:3] else 'Holdout',n=len(error),RMSE=np.sqrt(np.mean(error**2)),bias=error.mean(),censored_RMSE=np.sqrt(np.mean(censored**2)),area_pred_g_L=np.trapz(prediction,o.time_h),area_observed_g_L=np.trapz(o.co2_raw_g_l_h,o.time_h),peak=float(pred.max()),peak_h=float(t[pred.argmax()])))
  controls.append(dict(lab=lab,case=name,pre_nutrition_max_difference=prior_error,balance_max_error=abs(balance).max()))
metrics=pd.DataFrame(rows)
objective=sum(metrics.loc[(metrics.case=='Histórica')&metrics.lab.isin(LABS[:3]),'RMSE']**2)
assert np.isclose(objective,stored['objective'],atol=1e-8),(objective,stored['objective'])
print('Referencia histórica reproducida: objetivo',objective)
display(metrics.round(5))
display(pd.DataFrame(controls).round(10))


In [ ]:
def panels(title):
 fig,axes=plt.subplots(2,3,figsize=(16,9))
 fig.suptitle(title+'\nParámetros fijos · sin reajuste',y=.99)
 fig.subplots_adjust(top=.81,hspace=.37,wspace=.25)
 return fig,axes
for alternative,color in [('Sólo con nutrición','#D55E00'),('Sólo sin nutrición','#009E73')]:
 fig,axes=panels('CO₂ emitido · '+alternative+' frente a histórica')
 for ax,lab in zip(axes.flat,LABS):
  o=observed[lab];old=curves['Histórica',lab];alt=curves[alternative,lab]
  ax.plot(o.time_h,o.co2_raw_g_l_h,color='black',lw=1.1,label='Medianas horarias del sensor')
  ax.plot(old.index,old.qpred,color='#0072B2',label='Histórica actual')
  ax.plot(alt.index,alt.qpred,color=color,label=alternative)
  for ti,d in v.batches[lab].pulses['N']:ax.axvline(ti,color='gray',ls=':',lw=.8)
  role='Calibración previa' if lab in LABS[:3] else 'Holdout'
  ax.set(title=lab+' · '+role,xlabel='Tiempo desde inoculación [h]',ylabel='CO₂ [g/(L·h)]',xlim=(0,old.index[-1]));ax.set_ylim(bottom=0);ax.grid(alpha=.25)
 handles,labels=axes.flat[0].get_legend_handles_labels();fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.93),ncol=3,title='Datos y fuentes · líneas verticales: nutrición')
 plt.show()
for variable,title,ylabel in [('qprod','Producción que entra al depósito','CO₂ [g/(L·h)]'),('Cd','CO₂ disuelto','Cd [g/L]')]:
 fig,axes=panels(title+' · comparación de fuentes')
 for ax,lab in zip(axes.flat,LABS):
  for name,color in [('Histórica','#0072B2'),('Sólo con nutrición','#D55E00'),('Sólo sin nutrición','#009E73')]:
   f=curves[name,lab];ax.plot(f.index,f[variable],label=name,color=color)
  for ti,d in v.batches[lab].pulses['N']:ax.axvline(ti,color='gray',ls=':',lw=.8)
  role='Calibración previa' if lab in LABS[:3] else 'Holdout'
  ax.set(title=lab+' · '+role,xlabel='Tiempo desde inoculación [h]',ylabel=ylabel,xlim=(0,f.index[-1]));ax.set_ylim(bottom=0);ax.grid(alpha=.25)
 handles,labels=axes.flat[0].get_legend_handles_labels();fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.93),ncol=3,title='Fuentes · líneas verticales: nutrición')
 plt.show()
display(metrics.groupby(['role','case'])[['RMSE','bias','area_pred_g_L']].mean().round(5))
assert hashes()==before,'Se modificó un archivo protegido'
print('Verificado: modelos, parámetros, resultados previos y notebooks anteriores intactos. Sin exportar CSV/PNG.')


## Resultado y alcance

La referencia reproduce exactamente el objetivo histórico guardado: 0.0586968270. Las diferencias previas a la primera nutrición son inferiores a 1.4×10⁻⁷ g/(L·h); las diferencias posteriores proceden del cambio de fuente. Los balances del depósito se conservan.

Promedio de RMSE por laboratorio [g/(L·h)]:

| Grupo | Histórica | Sólo con nutrición | Sólo sin nutrición |
|---|---:|---:|---:|
| LAB019–021, calibración previa | 0.12706 | 0.19395 | 0.18329 |
| LAB016–018, holdout | 0.12004 | 0.21656 | 0.14033 |

Las dos alternativas aumentan el área emitida. En LAB019–021, el área media histórica es 37.85 g/L; las trayectorias únicas predicen 57.38 y 55.94 g/L. Esto evidencia que la combinación histórica reduce sustancialmente la fuente posterior a nutrición con estos coeficientes. Sólo sin nutrición mejora ligeramente el RMSE de LAB019, pero empeora el promedio de ambos grupos y mantiene sesgo positivo de cantidad.

Esta es una ablación con parámetros congelados, no una comparación entre formulaciones recalibradas. Sustituir el corchete completo también elimina el factor de actividad contenido dentro de él; no prueba por separado que sea necesario disponer de dos trayectorias. Se conserva el multiplicador transitorio externo incluso para q₀, de acuerdo con mantener el resto fijo. No se atribuye la ventaja estadística a un mecanismo biológico demostrado.
